# Sales Performance Analysis
**Q1–Q4 2025 | Regional & Product Overview**

This notebook explores quarterly sales trends, regional performance, and product category breakdowns using synthetic data.

## 1. Install Dependencies

Run the cell below once to install all required packages.

In [ ]:
%pip install pandas numpy matplotlib seaborn --quiet

: 

## 2. Setup & Sample Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.gridspec import GridSpec

PALETTE    = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3']
BG_COLOR   = '#F8F9FA'
GRID_COLOR = '#E0E0E0'

plt.rcParams.update({
    'figure.facecolor':  BG_COLOR,
    'axes.facecolor':    BG_COLOR,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.color':        GRID_COLOR,
    'grid.linewidth':    0.8,
    'font.family':       'sans-serif',
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.titleweight':  'bold',
    'axes.labelsize':    11,
})

rng      = np.random.default_rng(42)
quarters = ['Q1', 'Q2', 'Q3', 'Q4']
regions  = ['North', 'South', 'East', 'West']
products = ['Software', 'Hardware', 'Services', 'Support', 'Training']

revenue = pd.DataFrame(
    rng.integers(200, 800, size=(4, 4)) + np.array([0, 50, 120, 200]),
    index=quarters, columns=regions
)

months = pd.date_range('2025-01', periods=12, freq='MS')
trend  = pd.Series(
    3_000 + np.cumsum(rng.normal(80, 120, 12)) + 400 * np.sin(np.linspace(0, 2*np.pi, 12)),
    index=months
)

product_rev = pd.Series(
    rng.integers(300, 1200, len(products)), index=products
).sort_values(ascending=False)

csat = pd.DataFrame(
    rng.uniform(6.5, 9.5, size=(4, 4)).round(1),
    index=quarters, columns=regions
)

print('Data ready.')
revenue

## 3. Monthly Revenue Trend

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

ax.fill_between(trend.index, trend, alpha=0.15, color=PALETTE[0])
ax.plot(trend.index, trend, color=PALETTE[0], linewidth=2.5, marker='o', markersize=5)

peak_idx = trend.idxmax()
ax.annotate(
    f'Peak\n${trend[peak_idx]:,.0f}k',
    xy=(peak_idx, trend[peak_idx]),
    xytext=(20, -35), textcoords='offset points',
    arrowprops=dict(arrowstyle='->', color='#555'),
    fontsize=9, color='#333'
)

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}k'))
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b'))
ax.set_title('Monthly Revenue — FY 2025')
ax.set_xlabel('')
fig.tight_layout()
plt.show()

## 4. Regional Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

revenue.plot(
    kind='bar', ax=axes[0],
    color=PALETTE, width=0.7, edgecolor='white', linewidth=0.5
)
axes[0].set_title('Quarterly Revenue by Region')
axes[0].set_xlabel('Quarter')
axes[0].set_ylabel('Revenue ($k)')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Region', framealpha=0.7, fontsize=9)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}k'))

totals = revenue.sum().sort_values()
bars   = axes[1].barh(totals.index, totals.values, color=PALETTE[:len(totals)], edgecolor='white')
for bar, val in zip(bars, totals.values):
    axes[1].text(val + 15, bar.get_y() + bar.get_height()/2,
                 f'${val:,}k', va='center', fontsize=9)
axes[1].set_title('Total Annual Revenue by Region')
axes[1].set_xlabel('Revenue ($k)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}k'))
axes[1].grid(axis='y', alpha=0)

fig.tight_layout(pad=3)
plt.show()

## 5. Product Mix & Customer Satisfaction

In [ ]:
fig = plt.figure(figsize=(13, 5), facecolor=BG_COLOR)
gs  = GridSpec(1, 2, figure=fig, wspace=0.35)

ax1 = fig.add_subplot(gs[0])
wedges, texts, autotexts = ax1.pie(
    product_rev,
    labels=product_rev.index,
    autopct='%1.1f%%',
    colors=PALETTE,
    startangle=90,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2),
    pctdistance=0.75
)
for t in autotexts:
    t.set_fontsize(8.5)
    t.set_color('white')
    t.set_fontweight('bold')
ax1.set_title('Product Revenue Mix', pad=14)
total = product_rev.sum()
ax1.text(0, 0, f'${total:,}k\ntotal', ha='center', va='center',
         fontsize=10, fontweight='bold', color='#333')

ax2 = fig.add_subplot(gs[1])
sns.heatmap(
    csat, ax=ax2,
    annot=True, fmt='.1f', annot_kws={'size': 11, 'weight': 'bold'},
    cmap='RdYlGn', vmin=6, vmax=10,
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Score (1-10)', 'shrink': 0.8}
)
ax2.set_title('Customer Satisfaction by Quarter & Region')
ax2.set_xlabel('Region')
ax2.set_ylabel('Quarter')
ax2.tick_params(axis='both', rotation=0)

fig.suptitle('Product & Customer Insights — FY 2025', y=1.02, fontsize=14, fontweight='bold')
plt.show()

## 6. Summary Statistics

In [ ]:
summary = pd.DataFrame({
    'Total Revenue ($k)': revenue.sum(),
    'Best Quarter':        revenue.idxmax(),
    'Avg CSAT':            csat.mean().round(2),
    'Peak CSAT':           csat.max().round(2),
})

summary.style \
    .background_gradient(subset=['Total Revenue ($k)'], cmap='Blues') \
    .background_gradient(subset=['Avg CSAT', 'Peak CSAT'], cmap='RdYlGn', vmin=6, vmax=10) \
    .set_caption('Regional Summary — FY 2025') \
    .set_table_styles([{'selector': 'caption', 'props': 'font-size: 13px; font-weight: bold;'}])